In [ ]:
# https://docs.pytorch.org/tutorials/intermediate/dist_tuto.html

In [ ]:
# Next, code this out: https://docs.pytorch.org/tutorials/intermediate/dist_tuto.html#distributed-training

In [ ]:
import random
from torchvision import datasets, transforms
import math
import torch

In [ ]:
class WorkerDataset:
  def __init__(self, data, indices):
    self.data = data
    self.indices = indices

  def __len__(self):
    return len(self.indices)

  def __getitem__(self, index):
    data_idx = self.indices[index]
    return self.data[data_idx]

In [ ]:
class FullDataset:
  def __init__(self, data, world_size, seed=42):
    self.data = data
    self.world_size = world_size
    self.seed = seed
    self.part_indices = []

    all_indices = [i for i in range(len(data))]
    random.seed(self.seed)
    random.shuffle(all_indices)

    partition_len = math.ceil(len(all_indices) / self.world_size)
    for i in range(0, len(all_indices), partition_len):
      start_idx = i
      end_idx = i + partition_len
      self.part_indices.append(all_indices[start_idx:end_idx])

  def register(self, rank):
    return WorkerDataset(self.data, self.part_indices[rank])

In [ ]:
def partiton_dataset(dataset, rank, world_size):
  full_dataset = FullDataset(dataset, world_size)
  worker_dataset = full_dataset.register(rank=rank)
  train_set = torch.utils.data.DataLoader(
      worker_dataset,
      batch_size=128 // world_size,
      shuffle=True
  )
  return train_set

In [ ]:
dataset = datasets.MNIST('./data', train=True, download=True,
                             transform=transforms.Compose([
                                 transforms.ToTensor(),
                                 transforms.Normalize((0.1307,), (0.3081,))
                             ]))

In [ ]:
train_set = partiton_dataset(dataset, 0, 8)

In [ ]:
# model = Net()
# optimizer = optim.SGD(model.parameters(),
#                           lr=0.01, momentum=0.5)

# for data, target in train_set:
#   optimizer.zero_grad()
#   preds = Net(data)
#   loss = F.nll_loss(preds, target)
#   loss.backward()
#   average_gradients(model)
#   optimizer.step()

In [ ]:
import torch.distributed as dist

def average_gradients(model):
  size = float(dist.get_world_size())
  for param in model.parameters():
    dist.all_reduce(param.grad.data, op=dist.ReduceOp.AVG)

In [ ]:
# Next: implement Ring All-Reduce
# https://docs.pytorch.org/tutorials/intermediate/dist_tuto.html#our-own-ring-allreduce

In [1]:
## Ring-Allreduce

In [ ]:
def allreduce(send, recv):
  """
  It takes a recv tensor and will store the sum of all send tensors in it.
  Each GPU gets the same sum
  """

  rank = dist.get_rank()
  size = dist.get_world_size()

  send_buf = send.clone()
  recv_buf = send.clone()
  accum = send.clone()

  left = (rank - 1 + size) % size
  right = (rank + 1) % size

  for i in range(size - 1):
    if i % 2 == 0:
      # Send send_buf to next rank and receive the same from prev rank and accum
      _ = dist.isend(send_buf, right) # send my send_buf (non blocking) to the next rank 0 -> 1, 1 -> 2, etc
      dist.recv(recv_buf, left) # receive (blocking) from prev rank and store into my recv_buffer
      accum[:] += recv_buf
    else:
      # Send rec_buf (which you received before) to next rank and receive into send_buf and accum
      _ = dist.isend(recv_buf, right)
      dist.recv(send_buf, left)
      accum[:] += send_buf

    _.wait()

  recv[:] = accum[:]